In [ ]:
import json
import os
import polars as pl
import time
import tqdm.notebook as tqdm
import zipfile

from concurrent.futures import ThreadPoolExecutor
from google import genai
from pydantic import BaseModel
from typing import Literal


if not os.path.exists('genai_api_key.txt'):
    API_KEY = ''
else:
    with open('genai_api_key.txt', 'r') as f:
        API_KEY = f.read().strip()


instruct_model = 'gemini-2.5-flash-lite'
instruction = '''
Extract the key points from the following article summary related to AI:
- organization: The name of the organization or company that uses or affected by AI;
- industry: The industry this article is related to;
- impact: The impact of AI is: strong positive, positive, neutral, negative, strong negative;
- technology: The specific AI technology mentioned in the article; if not specific, return "AI".;
'''.strip() + "\n\n"
ai_client = genai.Client(api_key=API_KEY)


class KeyExtraction(BaseModel):
    organization: str
    industry: str
    impact: Literal[
        'strong positive',
        'positive',
        'neutral',
        'negative',
        'strong negative'
    ]
    technology: str


resp_config = genai.types.GenerateContentConfig(
    thinking_config=genai.types.ThinkingConfig(
        thinking_budget=0,
    ),
    response_mime_type="application/json",
    response_schema=KeyExtraction.model_json_schema()
)


def extract_key_points(text: str) -> KeyExtraction:
    response = ai_client.models.generate_content(
        model=instruct_model,
        contents=instruction + text,
        config=resp_config
    )
    return KeyExtraction.model_validate_json(response.text)


def safe_extract_key_points(text: str) -> dict:
    for _ in range(5):  # Retry up to 5 times
        try:
            resp = extract_key_points(text)
            return {"error": '', "res": resp.model_dump()}
        except Exception as e:
            last_exception = e
            time.sleep(.2)  # Backoff before retrying
    return {"error": str(last_exception), "res": None}

In [ ]:
resp_cache_dir = 'ner'
os.makedirs(resp_cache_dir, exist_ok=True)

def extract_key_points_and_save_batch(texts: list[str], idx: int) -> list[dict]:
    with ThreadPoolExecutor(max_workers=48) as executor:
        results = list(executor.map(safe_extract_key_points, texts))
    with open(os.path.join(resp_cache_dir, f'batch_{str(idx).zfill(3)}.jsonl'), 'w') as f:
        for res in results:
            f.write(json.dumps(res) + '\n')

article_df = pl.read_parquet('011_summary.parquet')
print(f'Number of articles: {article_df.height}')
texts = article_df.get_column("summary").to_list()

# Uncomment the following to reproduce the key points extraction and caching process.
# This is commented out to avoid unnecessary API calls during development/testing.

# BATCH_SIZE = 1000
# for i in tqdm.tqdm(list(range(50000, len(texts), BATCH_SIZE))):
#     extract_key_points_and_save_batch(texts[i:i+BATCH_SIZE], i // BATCH_SIZE)

In [ ]:
# store cache
with zipfile.ZipFile('ner.zip', 'w') as zipf:
    for root, dirs, files in os.walk(resp_cache_dir):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

In [ ]:
# Load from cache

with zipfile.ZipFile('ner.zip', 'r') as zipf:
    zipf.extractall(resp_cache_dir)

cache_files = sorted(list(os.listdir(resp_cache_dir)))
extraction_responses = []
error_count = 0

for file in cache_files:
    with open(os.path.join(resp_cache_dir, file), 'r') as f:
        for line in f:
            res = json.loads(line)
            if res["error"]:
                error_count += 1
            extraction_responses.append(res)

print(f"Total responses: {len(extraction_responses)}, Errors: {error_count}")

# Total responses: 164091, Errors: 1
# The only error here is because of model moderation. See `texts[49557]`.

## Merge dataset

In [ ]:
dates = article_df.get_column("date").to_list()
titles = article_df.get_column("title").to_list()

data = []
for idx, res in enumerate(extraction_responses):
    if res["error"]:
        continue
    data.append({
        "date": dates[idx],
        "title": titles[idx],
        "summary": texts[idx],
        **res["res"]
    })

df = pl.DataFrame(data)
impact_map = {
    'strong positive': 2,
    'positive': 1,
    'neutral': 0,
    'negative': -1,
    'strong negative': -2
}

df = df.with_columns(
    pl.col("impact").replace_strict(impact_map, return_dtype=pl.Int8).alias("impact")
)
df.write_parquet('020_labeled_data.parquet')
df.head()